# Sesión 04 – HDFS + formatos y diseño de almacenamiento

## Propósito
Diseñar el almacenamiento de un pipeline batch usando formato columnar y particionado, leyendo y escribiendo datos desde la zona de trabajo del laboratorio para simular el flujo de almacenamiento distribuido.

## Producto
Dataset almacenado en Parquet con particionado y una justificación técnica del diseño elegido.

## Contexto de la sesión
En la sesión anterior ya construimos el ETL con Spark.  
Ahora toca cerrar la unidad desde el lado del almacenamiento: **cómo queda organizada la salida del pipeline, cómo se vuelve a leer y qué decisiones de diseño ayudan a la performance**.

> En este laboratorio todavía no se está desplegando un HDFS real con NameNode y DataNode.  
> Sin embargo, trabajaremos la lógica correcta del flujo: **leer datos desde la zona de almacenamiento, procesarlos con Spark y persistir la salida en Parquet con particionado**.

## Paso 1: Crear la SparkSession

Inicializamos el entorno de trabajo en modo local dentro del contenedor Docker.  
El puerto `4040` queda disponible para revisar la interfaz de Spark si se desea observar jobs, stages y tareas.

También fijamos una cantidad moderada de particiones de shuffle para que la práctica sea más predecible.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-hdfs-formatos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/05 15:17:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Paso 2:  Revisar el dataset

El dataset fue diseñado con algunos **errores intencionales** para que el estudiante no solo ejecute código, sino que también detecte problemas de calidad y tome decisiones antes de almacenar la salida final.

## Paso 3: Leer los datasets desde la zona de datos

En esta práctica la carpeta `/opt/data` representa la **zona de almacenamiento de entrada** del laboratorio.

Desde la perspectiva del flujo Big Data, la idea es esta:

1. el dato vive en una zona de almacenamiento,
2. Spark lo lee,
3. lo transforma,
4. y publica una salida optimizada.

Por eso aquí sí estamos trabajando la lógica de “leer desde almacenamiento y procesar con Spark”, pero sin adelantar una infraestructura HDFS real.

In [2]:
ventas = spark.read.csv("/opt/data/ventas_s4.csv", header=True, inferSchema=True)
clientes = spark.read.csv("/opt/data/clientes_s4.csv", header=True, inferSchema=True)

ventas.show(5, truncate=False)
clientes.show(5, truncate=False)

+--------+----------+----------+-----------+------------------+-----+------+------+
|id_venta|cliente_id|fecha     |categoria  |producto          |monto|canal |region|
+--------+----------+----------+-----------+------------------+-----+------+------+
|1       |101       |2026-01-01|Lacteos    |Leche Entera      |120.5|Tienda|Sur   |
|2       |102       |2026-01-01|Lacteos    |Yogurt            |80.0 |Web   |Sur   |
|3       |103       |2026-01-02|Quesos     |Queso Fresco      |210.0|Tienda|Norte |
|4       |101       |2026-01-02|Lacteos    |Leche Deslactosada|140.0|Web   |Sur   |
|5       |104       |2026-01-02|Mantequilla|Mantequilla Barra |95.0 |Tienda|Centro|
+--------+----------+----------+-----------+------------------+-----+------+------+
only showing top 5 rows
+----------+---------+------------+--------+
|cliente_id|cliente  |tipo_cliente|segmento|
+----------+---------+------------+--------+
|101       |Cliente A|Minorista   |Oro     |
|102       |Cliente B|Minorista   |Plata

## Paso 4: Explorar estructura y tipos de datos

Antes de transformar y almacenar, verificamos esquema, tipos y volumen básico.

Esto responde una pregunta importante de ingeniería:

> ¿Estoy leyendo el dato con la estructura esperada antes de diseñar la salida?

Si el esquema ya viene mal desde el inicio, el problema se arrastra al join, al particionado y al almacenamiento final.

In [3]:
print("Total de registros en ventas:", ventas.count())
print("Total de registros en clientes:", clientes.count())

print("\nEsquema de ventas")
ventas.printSchema()

print("\nEsquema de clientes")
clientes.printSchema()

Total de registros en ventas: 33
Total de registros en clientes: 13

Esquema de ventas
root
 |-- id_venta: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- region: string (nullable = true)


Esquema de clientes
root
 |-- cliente_id: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- segmento: string (nullable = true)



## Paso 5: Exploración rápida de calidad

Antes de limpiar, revisamos los primeros síntomas:

- valores nulos,
- montos inválidos,
- posibles duplicados,
- dimensiones repetidas.

No se trata todavía de corregir; primero buscamos **entender el problema**.

In [4]:
ventas.show(10, truncate=False)

+--------+----------+----------+-----------+------------------+-----+------------+------+
|id_venta|cliente_id|fecha     |categoria  |producto          |monto|canal       |region|
+--------+----------+----------+-----------+------------------+-----+------------+------+
|1       |101       |2026-01-01|Lacteos    |Leche Entera      |120.5|Tienda      |Sur   |
|2       |102       |2026-01-01|Lacteos    |Yogurt            |80.0 |Web         |Sur   |
|3       |103       |2026-01-02|Quesos     |Queso Fresco      |210.0|Tienda      |Norte |
|4       |101       |2026-01-02|Lacteos    |Leche Deslactosada|140.0|Web         |Sur   |
|5       |104       |2026-01-02|Mantequilla|Mantequilla Barra |95.0 |Tienda      |Centro|
|6       |105       |2026-01-03|Quesos     |Queso Andino      |260.0|Distribuidor|Norte |
|7       |106       |2026-01-03|Lacteos    |Yogurt Griego     |160.0|Web         |Centro|
|8       |107       |2026-01-03|Lacteos    |Leche Entera      |110.0|Tienda      |Sur   |
|9       |

## Paso 6: Detectar errores intencionales en el dataset

En esta parte el estudiante debe identificar los problemas que afectarían al pipeline:

- un `cliente_id` vacío,
- un `monto` negativo,
- registros repetidos a nivel de negocio,
- un cliente duplicado en la tabla de referencia.

Este paso es clave porque **no todo problema de calidad se resuelve igual**.

In [5]:
print("Ventas con cliente_id nulo o vacío")
ventas.filter("cliente_id IS NULL OR cliente_id = ''").show(truncate=False)

print("Ventas con monto negativo")
ventas.filter("monto < 0").show(truncate=False)

print("Posibles duplicados exactos de negocio en ventas")
ventas.groupBy(
    "cliente_id", "fecha", "categoria", "producto", "monto", "canal", "region"
).count().filter("count > 1").show(truncate=False)

print("Clientes duplicados en la dimensión")
clientes.groupBy("cliente_id").count().filter("count > 1").show(truncate=False)

Ventas con cliente_id nulo o vacío


{"ts": "2026-04-05 15:24:02.939", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o50.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== SQL (line 1, position 23) ==\ncliente_id IS NULL OR cliente_id = ''\n                      ^^^^^^^^^^^^^^^\n\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNum

NumberFormatException: [CAST_INVALID_INPUT] The value '' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== SQL (line 1, position 23) ==
cliente_id IS NULL OR cliente_id = ''
                      ^^^^^^^^^^^^^^^


🔎 **Pregunta guiada**

Antes de continuar, responde:

1. ¿Qué registros sí deben eliminarse obligatoriamente?
2. ¿Qué registros podrían tratarse de otra forma según el negocio?
3. ¿Por qué conviene detectar duplicados en la dimensión antes de hacer el join?

## Paso 7: Limpieza básica para construir una salida confiable

Aplicamos una limpieza mínima orientada a esta práctica:

- eliminamos registros sin `cliente_id`, `fecha` o `monto`,
- filtramos montos negativos,
- deduplicamos clientes por `cliente_id`.

En ventas, además de revisar duplicados por negocio, generaremos una salida sin duplicados exactos por todas las columnas relevantes.

In [6]:
from pyspark.sql.functions import col

ventas_limpio = (
    ventas
    .dropna(subset=["cliente_id", "fecha", "monto"])
    .filter(col("monto") > 0)
    .dropDuplicates([
        "cliente_id", "fecha", "categoria", "producto", "monto", "canal", "region"
    ])
)

clientes_limpio = clientes.dropDuplicates(["cliente_id"])

print("Ventas limpias:", ventas_limpio.count())
print("Clientes limpios:", clientes_limpio.count())

Ventas limpias: 30
Clientes limpios: 12


## Paso 8: Verificar el efecto de la limpieza

Después de limpiar, volvemos a revisar los DataFrames para comprobar que:

- desaparecieron los nulos críticos,
- ya no hay montos inválidos,
- la dimensión de clientes quedó estable.

Este control evita seguir avanzando sobre supuestos incorrectos.

In [7]:
ventas_limpio.show(10, truncate=False)
clientes_limpio.show(10, truncate=False)

+--------+----------+----------+-----------+------------------+-----+------------+------+
|id_venta|cliente_id|fecha     |categoria  |producto          |monto|canal       |region|
+--------+----------+----------+-----------+------------------+-----+------------+------+
|14      |111       |2026-01-05|Mantequilla|Mantequilla Barra |98.0 |Tienda      |Sur   |
|20      |107       |2026-01-07|Lacteos    |Leche Entera      |118.0|Tienda      |Sur   |
|28      |103       |2026-01-10|Mantequilla|Mantequilla Light |132.0|Web         |Norte |
|4       |101       |2026-01-02|Lacteos    |Leche Deslactosada|140.0|Web         |Sur   |
|9       |108       |2026-01-04|Quesos     |Queso Paria       |300.0|Distribuidor|Sur   |
|18      |105       |2026-01-07|Quesos     |Queso Paria       |310.0|Distribuidor|Sur   |
|22      |109       |2026-01-08|Lacteos    |Yogurt            |82.0 |Tienda      |Centro|
|30      |105       |2026-01-10|Lacteos    |Leche Deslactosada|150.0|Web         |Norte |
|3       |

## Paso 9: Join distribuido entre ventas y clientes

Ahora integramos la información transaccional con la información descriptiva del cliente.

Este join tiene sentido de negocio porque nos permite responder preguntas como:

- ¿qué segmento compra más?
- ¿qué tipo de cliente concentra mayor monto?
- ¿qué regiones tienen mejor desempeño por tipo de cliente?

Además, esta tabla integrada será la base del almacenamiento final en Parquet.

In [8]:
df_join = ventas_limpio.join(clientes_limpio, "cliente_id", "inner")

df_join.show(10, truncate=False)
print("Total después del join:", df_join.count())

+----------+--------+----------+-----------+------------------+-----+------------+------+---------+------------+--------+
|cliente_id|id_venta|fecha     |categoria  |producto          |monto|canal       |region|cliente  |tipo_cliente|segmento|
+----------+--------+----------+-----------+------------------+-----+------------+------+---------+------------+--------+
|111       |14      |2026-01-05|Mantequilla|Mantequilla Barra |98.0 |Tienda      |Sur   |Cliente K|Mayorista   |Oro     |
|107       |20      |2026-01-07|Lacteos    |Leche Entera      |118.0|Tienda      |Sur   |Cliente G|Minorista   |Plata   |
|103       |28      |2026-01-10|Mantequilla|Mantequilla Light |132.0|Web         |Norte |Cliente C|Mayorista   |Oro     |
|101       |4       |2026-01-02|Lacteos    |Leche Deslactosada|140.0|Web         |Sur   |Cliente A|Minorista   |Oro     |
|108       |9       |2026-01-04|Quesos     |Queso Paria       |300.0|Distribuidor|Sur   |Cliente H|Mayorista   |Oro     |
|105       |18      |202

## Paso 10: Preparar columnas para particionado

Para almacenar con criterio, no basta con guardar el archivo.  
Primero debemos preparar columnas útiles para organizar la salida.

En esta práctica derivamos:

- `anio`
- `mes`

Estas columnas ayudarán a estructurar la carpeta de salida y a optimizar consultas frecuentes por periodo.

In [9]:
from pyspark.sql.functions import to_date, year, month

df_final = (
    df_join
    .withColumn("fecha", to_date("fecha"))
    .withColumn("anio", year("fecha"))
    .withColumn("mes", month("fecha"))
)

df_final.printSchema()
df_final.show(10, truncate=False)

root
 |-- cliente_id: integer (nullable = true)
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- region: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)

+----------+--------+----------+-----------+------------------+-----+------------+------+---------+------------+--------+----+---+
|cliente_id|id_venta|fecha     |categoria  |producto          |monto|canal       |region|cliente  |tipo_cliente|segmento|anio|mes|
+----------+--------+----------+-----------+------------------+-----+------------+------+---------+------------+--------+----+---+
|111       |14      |2026-01-05|Mantequilla|Mantequilla Barra |98.0 |Tienda      |Sur   |Clie

## Paso 11: Antes de escribir, pensar como ingenieros de datos

Aquí viene la parte central de la sesión.

La pregunta ya no es solo:

> ¿cómo guardo el DataFrame?

La pregunta correcta es:

> ¿cómo diseño el almacenamiento para que futuras consultas lean menos datos y respondan más rápido?

Por eso debemos justificar dos decisiones:

1. **Formato**
2. **Particionado**

## Paso 12: Comparar dos estrategias de salida

Vamos a escribir la misma salida en dos formatos:

- CSV
- Parquet

Luego compararemos tamaño y organización para discutir por qué Parquet es más adecuado en este contexto.

In [10]:
ruta_base = "/opt/artifacts/u1_s4"
ruta_csv = f"{ruta_base}/ventas_csv"
ruta_parquet = f"{ruta_base}/ventas_parquet"

import shutil
shutil.rmtree(ruta_base, ignore_errors=True)

(
    df_final.write
    .mode("overwrite")
    .option("header", True)
    .csv(ruta_csv)
)

print("CSV generado en:", ruta_csv)

CSV generado en: /opt/artifacts/u1_s4/ventas_csv


## Paso 13: Escribir la salida optimizada en Parquet con particionado

Ahora escribimos la salida en Parquet.

Elegimos particionar por:

- `anio`
- `mes`
- `region`

Esta propuesta es útil cuando las consultas suelen filtrar por periodo y región.  
Más adelante discutiremos también cuándo este diseño podría dejar de ser conveniente.

In [11]:
(
    df_final.write
    .mode("overwrite")
    .partitionBy("anio", "mes", "region")
    .parquet(ruta_parquet)
)

print("Parquet generado en:", ruta_parquet)

Parquet generado en: /opt/artifacts/u1_s4/ventas_parquet


## Paso 14: Leer nuevamente la salida Parquet

Este paso cierra el flujo lógico de la sesión:

1. el dato se leyó desde almacenamiento,
2. se procesó con Spark,
3. se persistió en formato optimizado,
4. y ahora se vuelve a leer como salida del pipeline.

Así el estudiante entiende que el almacenamiento no es un tema aislado: forma parte del ciclo completo de datos.

In [12]:
df_parquet = spark.read.parquet(ruta_parquet)

df_parquet.show(10, truncate=False)
print("Total registros en Parquet:", df_parquet.count())
df_parquet.printSchema()

+----------+--------+----------+-----------+------------------+-----+------------+---------+------------+--------+----+---+------+
|cliente_id|id_venta|fecha     |categoria  |producto          |monto|canal       |cliente  |tipo_cliente|segmento|anio|mes|region|
+----------+--------+----------+-----------+------------------+-----+------------+---------+------------+--------+----+---+------+
|111       |14      |2026-01-05|Mantequilla|Mantequilla Barra |98.0 |Tienda      |Cliente K|Mayorista   |Oro     |2026|1  |Sur   |
|107       |20      |2026-01-07|Lacteos    |Leche Entera      |118.0|Tienda      |Cliente G|Minorista   |Plata   |2026|1  |Sur   |
|101       |4       |2026-01-02|Lacteos    |Leche Deslactosada|140.0|Web         |Cliente A|Minorista   |Oro     |2026|1  |Sur   |
|108       |9       |2026-01-04|Quesos     |Queso Paria       |300.0|Distribuidor|Cliente H|Mayorista   |Oro     |2026|1  |Sur   |
|105       |18      |2026-01-07|Quesos     |Queso Paria       |310.0|Distribuidor|C

## Paso 15: Observar la estructura física de carpetas

Una ventaja del particionado es que el layout de carpetas expresa una organización útil para el acceso.

Aquí inspeccionamos la estructura que Spark creó automáticamente.  
Esto permite conectar el concepto lógico de `partitionBy()` con el resultado físico sobre almacenamiento.

In [13]:
import os

for root, dirs, files in os.walk(ruta_parquet):
    nivel = root.replace(ruta_parquet, "").count(os.sep)
    sangria = " " * 2 * nivel
    nombre = os.path.basename(root) if os.path.basename(root) else root
    print(f"{sangria}{nombre}/")
    for f in files[:4]:
        print(f"{sangria}  {f}")

ventas_parquet/
  ._SUCCESS.crc
  _SUCCESS
  anio=2026/
    mes=1/
      region=Centro/
        .part-00000-3467f96c-4b91-408a-b8d1-2894228cb896.c000.snappy.parquet.crc
        part-00000-3467f96c-4b91-408a-b8d1-2894228cb896.c000.snappy.parquet
      region=Norte/
        .part-00000-3467f96c-4b91-408a-b8d1-2894228cb896.c000.snappy.parquet.crc
        part-00000-3467f96c-4b91-408a-b8d1-2894228cb896.c000.snappy.parquet
      region=Sur/
        .part-00000-3467f96c-4b91-408a-b8d1-2894228cb896.c000.snappy.parquet.crc
        part-00000-3467f96c-4b91-408a-b8d1-2894228cb896.c000.snappy.parquet


## Paso 16: Consulta típica que se beneficia del particionado

Supongamos que un analista necesita consultar ventas de enero en la región Sur.

Si la salida está particionada por `anio`, `mes` y `region`, Spark puede reducir la lectura a una fracción del total.

Aquí no estamos midiendo todavía tiempos de benchmark avanzados, pero sí mostrando la lógica correcta de diseño.

In [14]:
from pyspark.sql.functions import col

consulta_sur = (
    df_parquet
    .filter((col("anio") == 2026) & (col("mes") == 1) & (col("region") == "Sur"))
    .select("fecha", "cliente", "producto", "monto", "region", "segmento")
)

consulta_sur.show(truncate=False)

+----------+---------+------------------+-----+------+--------+
|fecha     |cliente  |producto          |monto|region|segmento|
+----------+---------+------------------+-----+------+--------+
|2026-01-05|Cliente K|Mantequilla Barra |98.0 |Sur   |Oro     |
|2026-01-07|Cliente G|Leche Entera      |118.0|Sur   |Plata   |
|2026-01-02|Cliente A|Leche Deslactosada|140.0|Sur   |Oro     |
|2026-01-04|Cliente H|Queso Paria       |300.0|Sur   |Oro     |
|2026-01-07|Cliente E|Queso Paria       |310.0|Sur   |Plata   |
|2026-01-03|Cliente G|Leche Entera      |110.0|Sur   |Plata   |
|2026-01-04|Cliente A|Yogurt            |75.0 |Sur   |Oro     |
|2026-01-10|Cliente B|Queso Paria       |315.0|Sur   |Plata   |
|2026-01-01|Cliente A|Leche Entera      |120.5|Sur   |Oro     |
|2026-01-09|Cliente A|Leche Entera      |121.0|Sur   |Oro     |
|2026-01-01|Cliente B|Yogurt            |80.0 |Sur   |Plata   |
|2026-01-08|Cliente H|Queso Fresco      |205.0|Sur   |Oro     |
+----------+---------+------------------

## Paso 17: Comparación simple de tamaño entre CSV y Parquet

Una diferencia importante entre formatos es el tamaño final de almacenamiento.

Con datasets pequeños la diferencia puede no ser enorme, pero el ejercicio sigue siendo útil para introducir dos ideas:

- compresión,
- lectura selectiva por columnas.

En escenarios grandes, estas diferencias se vuelven mucho más relevantes.

In [15]:
def size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return round(total / (1024 * 1024), 6)

print("Tamaño CSV (MB):", size_mb(ruta_csv))
print("Tamaño Parquet (MB):", size_mb(ruta_parquet))

Tamaño CSV (MB): 0.00273
Tamaño Parquet (MB): 0.009791


## Paso 18: Comparación de lectura selectiva

Cuando se trabaja con CSV, normalmente se lee fila completa.  
Con Parquet, Spark puede aprovechar mejor la estructura columnar.

Aquí simulamos una lectura donde solo necesitamos unas pocas columnas para análisis.

In [16]:
df_parquet.select("fecha", "producto", "monto", "region").show(10, truncate=False)

+----------+------------------+-----+------+
|fecha     |producto          |monto|region|
+----------+------------------+-----+------+
|2026-01-05|Mantequilla Barra |98.0 |Sur   |
|2026-01-07|Leche Entera      |118.0|Sur   |
|2026-01-02|Leche Deslactosada|140.0|Sur   |
|2026-01-04|Queso Paria       |300.0|Sur   |
|2026-01-07|Queso Paria       |310.0|Sur   |
|2026-01-03|Leche Entera      |110.0|Sur   |
|2026-01-04|Yogurt            |75.0 |Sur   |
|2026-01-10|Queso Paria       |315.0|Sur   |
|2026-01-01|Leche Entera      |120.5|Sur   |
|2026-01-09|Leche Entera      |121.0|Sur   |
+----------+------------------+-----+------+
only showing top 10 rows


## Paso 19: Errores y decisiones que deben detectar los estudiantes

Durante esta práctica el estudiante debería poder reconocer al menos estos puntos:

1. había un `cliente_id` nulo,
2. había un `monto` negativo,
3. había duplicado lógico en ventas,
4. había duplicado en la tabla de clientes,
5. no todo particionado es bueno por defecto.

Esto convierte la sesión en una práctica de ingeniería y no solo en un ejercicio de sintaxis.

## Paso 20: Mini ficha técnica de cierre

Completa esta guía breve con base en lo que observaste en el laboratorio.

### Formato elegido
Parquet

### Particionado elegido
`anio / mes / region`

### Consulta objetivo
Ventas por periodo y región.

### Justificación
Parquet reduce lectura innecesaria por su estructura columnar, y el particionado ayuda cuando se filtra por tiempo y región.

### Riesgo
Si una columna de partición tiene cardinalidad muy alta o no se usa en filtros, se pueden crear demasiadas carpetas y empeorar la administración del almacenamiento.

## Paso 21: Reflexión final

**Preguntas para reflexión:**

- ¿Por qué esta sesión no se reduce a “guardar archivos”?
- ¿Qué relación existe entre el patrón de consulta y el particionado?
- ¿En qué caso elegirías otro esquema de partición?
- ¿Qué diferencia conceptual hay entre “zona de datos” y “formato de salida”?

In [ ]:
# Espacio opcional para respuestas o pruebas adicionales

# Actividad Autónoma – Aplicación al Producto de la Unidad 1

## Objetivo
Tomar la salida ETL de su caso de trabajo y proponer un diseño de almacenamiento justificando:

- formato,
- columnas de partición,
- consulta objetivo,
- riesgos del diseño.

## Entregable breve
1. Captura de la estructura de carpetas generada.
2. Evidencia de lectura desde Parquet.
3. Justificación del formato y particionado.
4. Una consulta que se beneficie del diseño propuesto.